# Description

# Modules loading

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import re
from pathlib import Path

import pandas as pd

from entity import Trait
import conf

# Settings

In [3]:
LV_NAME = "LV925"

# Paths

In [4]:
OUTPUT_FIGURES_DIR = Path(conf.RESULTS_DIR, "demo", f"{LV_NAME.lower()}").resolve()
display(OUTPUT_FIGURES_DIR)
OUTPUT_FIGURES_DIR.mkdir(parents=True, exist_ok=True)

PosixPath('/opt/data/results/demo/lv925')

# Load MultiPLIER summary

In [5]:
multiplier_model_summary = pd.read_pickle(conf.MULTIPLIER["MODEL_SUMMARY_FILE"])

In [6]:
multiplier_model_summary.shape

(2157, 5)

In [7]:
multiplier_model_summary.head()

,pathway,LV index,AUC,p-value,FDR
1,KEGG_LYSINE_DEGRADATION,1,0.388059,0.866078,0.956005
2,REACTOME_MRNA_SPLICING,1,0.733057,0.000048,0.000582
3,MIPS_NOP56P_ASSOCIATED_PRE_RRNA_COMPLEX,1,0.680555,0.001628,0.011366
4,KEGG_DNA_REPLICATION,1,0.549473,0.312155,0.539951
5,PID_MYC_ACTIVPATHWAY,1,0.639303,0.021702,0.083739


# LV pathways

In [8]:
lv_pathways = multiplier_model_summary[
    multiplier_model_summary["LV index"].isin((LV_NAME[2:],))
    & (
        (multiplier_model_summary["FDR"] < 0.05)
                | (multiplier_model_summary["AUC"] >= 0.75)
    )
]

In [9]:
lv_pathways.shape

(0, 5)

In [10]:
lv_pathways = lv_pathways[["pathway", "AUC", "FDR"]].sort_values("FDR")

In [11]:
lv_pathways = lv_pathways.assign(AUC=lv_pathways["AUC"].apply(lambda x: f"{x:.2f}"))

In [12]:
lv_pathways = lv_pathways.assign(FDR=lv_pathways["FDR"].apply(lambda x: f"{x:.2e}"))

In [13]:
lv_pathways = lv_pathways.rename(
    columns={
        "pathway": "Pathway",
    }
)

In [14]:
lv_pathways.head()

,Pathway,AUC,FDR


# Load LV data

In [15]:
from data.recount2 import LVAnalysis

In [16]:
lv_obj = LVAnalysis(LV_NAME)

Here I show the top 20 genes for our LV. You can see gene symbols, the LV weight (in column `LV603`) and the cytoband.

In [17]:
lv_obj.lv_genes.head(20)

,gene_name,LV925,gene_band
0,RAB26,3.962000,16p13.3
1,CREB3L4,3.809981,1q21.3
2,ZNF552,2.676678,19q13.43
3,LINC00921,2.494355,16p13.3
4,TACR2,2.144907,10q22.1
5,NPB,1.993333,NaN
6,C11orf80,1.978051,11q13.2
7,NOXO1,1.867241,NaN
8,BAMBI,1.734197,10p12.1
9,REM2,1.718219,14q11.2


# Pathway enrichment using external databases

## gProfiler

In [18]:
print(" ".join(lv_obj.lv_genes.head(70)["gene_name"].tolist()))

RAB26 CREB3L4 ZNF552 LINC00921 TACR2 NPB C11orf80 NOXO1 BAMBI REM2 ZKSCAN4 SETDB1 PARD6B WARS2 IL5 RPS6KB1 WRAP53 GGCX AP4S1 HES1 RNF43 SLC22A5 TGIF1 OAZ3 RFC4 NAT1 ZNF446 P2RY2 APPBP2 BRIP1 GADD45G PEX12 ESR1 TSEN34 IL23A COX6B2 ZNF595 SLC16A14 KCNK6 CHRNG CHRNB2 NTN3 PEX16 PSPN EFNA3 CNKSR1 HEXIM1 MRPL22 ST3GAL2 TAF12 ZNF263 SPSB2 SLC9A5 MAP3K12 GPATCH1 ZNF625 SLC10A1 NRAS KCNG2 IL18BP HIST1H3E SMAD6 PSMD6 SMPDL3B TRIM37 AP3S2 GP1BA L2HGDH NUDT4 RHOF


Copy/paste the list of genes above and use gProfiler: https://biit.cs.ut.ee/gprofiler/gost

Results URL: https://biit.cs.ut.ee/gplink/l/aXGV7WPJRT3

**Notes**: only one significant pathways; looks like an artifact.

## FUMA

In [19]:
# print top genes in module
print("\n".join(lv_obj.lv_genes.head(70)["gene_name"].tolist()))

RAB26
CREB3L4
ZNF552
LINC00921
TACR2
NPB
C11orf80
NOXO1
BAMBI
REM2
ZKSCAN4
SETDB1
PARD6B
WARS2
IL5
RPS6KB1
WRAP53
GGCX
AP4S1
HES1
RNF43
SLC22A5
TGIF1
OAZ3
RFC4
NAT1
ZNF446
P2RY2
APPBP2
BRIP1
GADD45G
PEX12
ESR1
TSEN34
IL23A
COX6B2
ZNF595
SLC16A14
KCNK6
CHRNG
CHRNB2
NTN3
PEX16
PSPN
EFNA3
CNKSR1
HEXIM1
MRPL22
ST3GAL2
TAF12
ZNF263
SPSB2
SLC9A5
MAP3K12
GPATCH1
ZNF625
SLC10A1
NRAS
KCNG2
IL18BP
HIST1H3E
SMAD6
PSMD6
SMPDL3B
TRIM37
AP3S2
GP1BA
L2HGDH
NUDT4
RHOF


In [20]:
# save all genes in model to use as background list of genes
lv_obj.lv_genes["gene_name"].to_csv(OUTPUT_FIGURES_DIR / "all_genes.txt", header=None, index=False)

In [21]:
OUTPUT_FIGURES_DIR / "all_genes.txt"

PosixPath('/opt/data/results/demo/lv925/all_genes.txt')

In [22]:
!head /opt/data/results/demo/lv24/all_genes.txt

KCNK7
KRT1
ACER1
ASPRV1
CDHR1
CTNNBIP1
CAPNS2
ELOVL3
KLC3
DSP


In [23]:
!wc -l /opt/data/results/demo/lv24/all_genes.txt

6750 /opt/data/results/demo/lv24/all_genes.txt


Now go to the FUMA GENE2FUNC module here: https://fuma.ctglab.nl/gene2func

1. Paste the list of top genes above and then upload the `all_genes.txt` file.
2. Use a "Title" and click on "Submit"

**Notes:** not specifically expressed in any tissue in GTEx. Another sign that this LV might be artifact (not interesting).